In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-20T14:33:10.599525+00:00', 'open': 101.12, 'high': 103.74, 'low': 100.32, 'close': 102.79, 'volume': 854, 'trade_count': 32, 'vwap': 102.24}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-20T14:33:10.599525+00:00', 'open': 95.37, 'high': 99.13, 'low': 94.48, 'close': 97.19, 'volume': 665, 'trade_count': 31, 'vwap': 97.46}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-20T14:33:10.599525+00:00', 'open': 100.77, 'high': 102.86, 'low': 100.88, 'close': 101.9, 'volume': 727, 'trade_count': 28, 'vwap': 102.46}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-20T14:33:11.619861+00:00', 'open': 106.67, 'high': 108.69, 'low': 105.09, 'close': 107.47, 'volume': 462, 'trade_count': 25, 'vwap': 107.52}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-20T14:33:11.619861+00:00', 'open': 97.66, 'high': 100.56, 'low': 97.44, 'close': 99.42, 'volume': 536, 'trade_count': 22, 'vwap': 99.56}
Pushed to 